### Step 1: Import Libraries & API Key

In [9]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import requests
import json
from IPython.display import display, Markdown

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()

### Step 2: Set up Pushover

Pushover is a tool that sends you a notification on your phone.

Step 2a -> Set up account in your browser
Step 2b -> Set up the app on your iPhone/Android; log into the same account
Step 2c -> In your browser, create an "Application/API Token"
Step 2d -> Copy your User Key (x) and API Token (y) into the .env file like this with your own keys:
PUSHOVER_USER = xxxxxxxxxxxxxxxxxxxx
PUSHOVER_TOKEN = yyyyyyyyyyyyyyyyyyy

**Then save changes to the .env file.**

Also run a test on https://pushover.net/ to manually send a notification from Pushover in your browser to pushover on your phone.

In [10]:
load_dotenv() #Rerun to load the changes you made (make sure you saved them (to the .env file) first)

True

In [11]:
# Get the values from the environment file (.env file):
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [7]:
# Check:
# Do this (in private!) to test that keys have been loaded.
# print(pushover_user)
# print(pushover_token)

In [12]:
# Test Pushover (send ourselves a notification)
def send_notification(message: str): #Created a function that expects a string
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data = payload)

In [13]:
#Test Run:
send_notification("Hello to myself, from this amazing AI Engineering training!")

### Step 3: Describe Pushover as an LLM Tool

In [14]:
# LLM needs to know what tools it has and the details about each.
# We need to provide the LLM a description of the send_notification function using JSON.

# This is a dictionary that describes the send_notification function
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information.",
    # tells LLM what this tool is and what it should be used for
    "parameters": {
        "type": "object", #Required to write this because LLM's pass parameters through a JSON object (inside which they'll be a string for this function (push notification example))
        "properties": {
            "message": { #The send_notification function has the property of message.
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"] #A required parameter is message. #List of things (parameters in this case) that are actually required.
    }

}


### Step 4: Add Pushover to the List of Tools for the LLM

In [15]:
tools = [{"type": "function", "function": send_notification_function}] #We give the LLM the function description.
# List of function dictionaries for various tools, such as sending notifications.
# We have a dictionary (send_notification_function) inside of a list of dictionaries (tools)

### Step 5: Calling the Tool from an LLM

In [16]:
response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages = [
    {"role":"user", "content": "Please send me a notification telling me what amazing progress\
     I'm making on the AI Engineering training by SuperDataScience."} #User Message. User gives LLM instructions to call a tool.
    # Telling the LLM explicitly to send us a notification.
    ],
    tools = tools, #Pass the tools the LLM can use, so the tools list
    #tool_choice = "auto" (Default) #Lets the LLM choose the tool on its own. If tool_choice = "none", then the LLM won't call any tools.
    # If tool_choice = "required", then it would have to call at least 1 tool. Or alternatively, you could give it a dictionary of tools it could choose from.
)

message = response.choices[0].message

**Recall:** You have to process in your **code** what to do if the LLM does do a tool call because it's up to your code to catch that.

In [19]:
# Check if model wants to call a tool
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_hnOKeoki3b0k5HmkupojNgmj', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!"}', name='send_notification'), type='function')])


*Output:*
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_eewMZu8nlpAE380jNOtYLwLC', function=Function(arguments='{"message":"You are making amazing progress on the AI Engineering training by SuperDataScience. Keep up the great work!"}', name='send_notification'), type='function')])

In [ ]:
#So the LLM is trying to call a tool as we have:
"""
tool_calls=[
    ChatCompletionMessageFunctionToolCall(
        id='call_eewMZu8nlpAE380jNOtYLwLC',
        function=Function(
            arguments='{
            "message":"You are making amazing progress on the AI Engineering training by SuperDataScience. \
                Keep up the great work!"
            }',
            name='send_notification'
        ),
        type='function'
    )
]
"""

In [20]:
# How the LLM Processes that:
#Tool Handling Code
if message.tool_calls: # So if there's something in there (in messages -> tool_call)
    tool_call = message.tool_calls[0] #Checks what is the element in there
    args = json.loads(tool_call.function.arguments) #Our arguments (extracting from tool_call, we are getting the arguments that were passed to the function, so message argument in our send_notification function (the message itself))

    #Actually send the notification now.
    send_notification(args["message"])
    print(f"Sent notification: {args['message']}")

else:
    print(message.content)

Sent notification: You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!


We get an AI-generated notification message on our phone! saying:
Sent notification: You are making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work!